# 🎬 Wan2.2 Video Generator - Google Colab

Generate AI videos using the Wan2.2 models on Google Colab's free GPU!

## 📋 Before You Start:

1. **Enable GPU**: Go to `Runtime` → `Change runtime type` → Select `T4 GPU` or `L4 GPU`
2. **Free Tier Limits**: Google Colab free tier has usage limits. Generation may take 10-20 minutes.
3. **Model Size**: We'll use TI2V-5B (smallest model, ~15GB download)

## 🚀 How to Use:

1. Run each cell in order (click the play button or press Shift+Enter)
2. Wait for the Gradio link at the end
3. Click the link to open the UI
4. Generate your videos!

---

## Step 1: Check GPU Availability

This cell checks if you have a GPU enabled.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU Detected: {gpu_name}")
    print(f"   Total VRAM: {gpu_memory:.1f} GB")
    if gpu_memory < 15:
        print("⚠️  Warning: Your GPU has less than 15GB VRAM. Generation may fail.")
        print("   Try upgrading to Colab Pro for better GPUs.")
else:
    print("❌ No GPU detected!")
    print("   Please enable GPU: Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU is required to run Wan2.2")

## Step 2: Install Dependencies

This will take 3-5 minutes. Installing flash_attn may show warnings - this is normal.

In [ ]:
%%capture
# Install packages silently (remove %%capture to see full output)

!pip install gradio>=4.0.0
!pip install torch>=2.4.0 torchvision>=0.19.0 torchaudio
!pip install opencv-python>=4.9.0.80
!pip install diffusers>=0.31.0
!pip install transformers>=4.49.0 tokenizers>=0.20.3
!pip install accelerate>=1.1.1 tqdm imageio[ffmpeg] easydict ftfy dashscope imageio-ffmpeg
!pip install 'numpy>=1.23.5,<2'

# Install flash_attn (may take longer)
!pip install flash_attn --no-build-isolation

print("✅ All dependencies installed!")

## Step 3: Clone Wan2.2 Repository

Download the Wan2.2 code from GitHub.

In [ ]:
import os

# Clone the repository
if not os.path.exists('Wan2.2'):
    !git clone https://github.com/Wan-Video/Wan2.2.git
    print("✅ Repository cloned!")
else:
    print("✅ Repository already exists!")

# Change to the directory
%cd Wan2.2

## Step 4: Download Model Checkpoint

**Choose ONE option below:**

### Option A: TI2V-5B (Recommended for Colab Free Tier)
- Size: ~15GB
- Works on T4 GPU (16GB VRAM)
- Supports both Text-to-Video and Image-to-Video
- Resolution: 720P

Run this cell to download TI2V-5B:

In [ ]:
# Download TI2V-5B model (recommended)
!pip install -U "huggingface_hub[cli]"

import os
if not os.path.exists('./Wan2.2-TI2V-5B'):
    print("📥 Downloading TI2V-5B model (~15GB, may take 10-15 minutes)...")
    !huggingface-cli download Wan-AI/Wan2.2-TI2V-5B --local-dir ./Wan2.2-TI2V-5B
    print("✅ Model downloaded successfully!")
else:
    print("✅ Model already downloaded!")

MODEL_PATH = './Wan2.2-TI2V-5B'
MODEL_TYPE = 'ti2v-5B'

### Option B: T2V-A14B (For Colab Pro+ with A100)

**⚠️ Only run this if you have Colab Pro+ with A100 GPU (80GB VRAM)**

Uncomment and run to download T2V-A14B:

In [ ]:
# # Uncomment to download T2V-A14B (requires A100 GPU)
# !pip install -U "huggingface_hub[cli]"
# 
# import os
# if not os.path.exists('./Wan2.2-T2V-A14B'):
#     print("📥 Downloading T2V-A14B model (~30GB+)...")
#     !huggingface-cli download Wan-AI/Wan2.2-T2V-A14B --local-dir ./Wan2.2-T2V-A14B
#     print("✅ Model downloaded successfully!")
# else:
#     print("✅ Model already downloaded!")
# 
# MODEL_PATH = './Wan2.2-T2V-A14B'
# MODEL_TYPE = 't2v-A14B'

## Step 5: Create Colab-Optimized UI

This creates a simplified Gradio interface optimized for Colab.

In [ ]:
%%writefile app_colab.py
#!/usr/bin/env python3
"""Wan2.2 Gradio UI - Google Colab Optimized"""

import os
import sys
import gradio as gr
import subprocess
from pathlib import Path

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

def run_generation(task, prompt, image_path, size, ckpt_dir, num_frames, seed, steps, guide_scale):
    """Run video generation"""
    
    if not ckpt_dir or not os.path.exists(ckpt_dir):
        return None, "Error: Checkpoint directory not found"
    
    cmd = [
        "python", "generate.py",
        "--task", task,
        "--size", size,
        "--ckpt_dir", ckpt_dir,
        "--prompt", prompt or "",
        "--offload_model", "True",
        "--convert_model_dtype",
        "--t5_cpu"
    ]
    
    if image_path and os.path.exists(image_path):
        cmd.extend(["--image", image_path])
    
    if num_frames:
        cmd.extend(["--frame_num", str(num_frames)])
    
    if seed >= 0:
        cmd.extend(["--base_seed", str(seed)])
    
    if steps:
        cmd.extend(["--sample_steps", str(steps)])
    
    if guide_scale:
        cmd.extend(["--sample_guide_scale", str(guide_scale)])
    
    try:
        print(f"Running command: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1200)
        
        output_files = sorted(Path('.').glob(f'{task}*.mp4'), key=os.path.getmtime, reverse=True)
        
        if output_files:
            video_path = str(output_files[0])
            log_message = f"✅ Generation successful!\n\nOutput: {video_path}\n\nStdout:\n{result.stdout[-2000:]}\n\nStderr:\n{result.stderr[-1000:]}"
            return video_path, log_message
        else:
            return None, f"❌ No output video found.\n\nStdout:\n{result.stdout[-2000:]}\n\nStderr:\n{result.stderr[-1000:]}"
    
    except subprocess.TimeoutExpired:
        return None, "❌ Generation timed out after 20 minutes"
    except Exception as e:
        return None, f"❌ Error: {str(e)}"

# Create Gradio Interface
with gr.Blocks(title="Wan2.2 - Colab", theme=gr.themes.Soft()) as demo:
    
    gr.Markdown("""
    # 🎬 Wan2.2 Video Generator (Colab)
    
    Generate AI videos using the Wan2.2 TI2V-5B model.
    
    **Tips:**
    - Leave image empty for Text-to-Video mode
    - Upload image for Image-to-Video mode
    - Generation takes 10-20 minutes on T4 GPU
    - Start with 81 frames (~3 seconds)
    """)
    
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(
                label="Prompt",
                placeholder="A butterfly flying through a sunny garden...",
                lines=3
            )
            image = gr.Image(label="Input Image (Optional - leave empty for T2V)", type="filepath")
            
            with gr.Row():
                size = gr.Dropdown(
                    choices=["1280*704", "704*1280"],
                    value="1280*704",
                    label="Video Size"
                )
                frames = gr.Slider(1, 161, value=81, step=4, label="Frames (81 = ~3 sec)")
            
            with gr.Row():
                seed = gr.Number(label="Seed (-1 = random)", value=-1, precision=0)
                steps = gr.Slider(10, 50, value=30, step=1, label="Steps")
                guide = gr.Slider(1.0, 15.0, value=7.5, step=0.5, label="Guidance")
            
            generate_btn = gr.Button("🎬 Generate Video", variant="primary", size="lg")
        
        with gr.Column():
            output_video = gr.Video(label="Generated Video")
            output_log = gr.Textbox(label="Log", lines=15)
    
    # Hidden components for config
    task = gr.State(value="ti2v-5B")
    ckpt_dir = gr.State(value="./Wan2.2-TI2V-5B")
    
    generate_btn.click(
        fn=run_generation,
        inputs=[task, prompt, image, size, ckpt_dir, frames, seed, steps, guide],
        outputs=[output_video, output_log]
    )
    
    gr.Markdown("""
    ---
    **Note:** First generation may be slower due to model loading.
    """)

if __name__ == "__main__":
    demo.launch(share=True, debug=True)

print("✅ Colab UI file created!")

## Step 6: Launch the UI! 🚀

This will start the Gradio interface. Look for the **public URL** (gradio.live link) in the output below.

Click the link to open the UI in a new tab!

In [ ]:
# Launch the UI
print("🚀 Launching Wan2.2 Web UI...")
print("⏳ This may take a minute to start...")
print("")
print("👇 Look for the 'public URL' link below and click it!")
print("")

!python app_colab.py

## 💡 Tips for Best Results

### For Text-to-Video:
- Leave the image field empty
- Write detailed prompts describing:
  - What's happening (action)
  - Camera movement
  - Lighting and mood
  - Style (cinematic, anime, realistic, etc.)

**Example Prompts:**
- "A majestic eagle soaring over snow-capped mountains at sunset, cinematic drone shot"
- "Neon-lit cyberpunk city street with rain, people walking with umbrellas, night scene"
- "Close-up of a flower blooming in timelapse, soft natural lighting, macro photography"

### For Image-to-Video:
- Upload a clear image
- Describe the motion you want to see
- Keep prompts focused on movement

**Example:**
- Image: Photo of a cat
- Prompt: "The cat's ears twitch, it looks around curiously, then meows"

### Frame Settings:
- **81 frames** = ~3 seconds (good for testing)
- **121 frames** = ~5 seconds (balanced)
- **161 frames** = ~7 seconds (longer, slower generation)

### If Generation Fails:
1. Try reducing frames to 81
2. Restart runtime: `Runtime` → `Restart runtime`
3. Re-run all cells
4. Check your GPU hasn't run out of quota

---

## 🔗 Resources

- [Wan2.2 GitHub](https://github.com/Wan-Video/Wan2.2)
- [Official Website](https://wan.video)
- [Research Paper](https://arxiv.org/abs/2503.20314)

---

**Enjoy generating AI videos! 🎬✨**